<a href="https://colab.research.google.com/github/ece7048/MHL-RGB-deep-learing/blob/main/noise_reduce_cells.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import subprocess
from typing import Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch import Tensor

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
!pip install torch_geometric
!pip install monai
!pip install wandb
!pip install quantus

In [ ]:
loada3='resnet50'
loada4='densenet121'
loada='vgg16'
loada2='vit'
PATH= "/content/gdrive/MyDrive/Colab/DATA_cell/trained_models/"+loada2

In [ ]:
import monai
from monai.networks.nets import DenseNet121, Classifier, EfficientNet, EfficientNetBN, ViT
from torchvision import transforms, datasets, models

class Net(nn.Module):

    def __init__(self) -> None:
        super(Net, self).__init__()
        self.conv1 = nn.Conv3d(1, 6, 5, 5)
        self.pool = nn.MaxPool3d(2, 2)
        self.conv2 = nn.Conv3d(6, 16, 2, 2)
        self.fc1 = nn.Linear(1152 , 512)
        self.fc2 = nn.Linear(512, 126)
        self.fc3 = nn.Linear(126, 2)
        #self.fc4 = nn.Linear(16, 2)

    def forward(self, x: Tensor) -> Tensor:
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.shape[0],x.shape[1]*x.shape[2]*x.shape[3]*x.shape[4])
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
       # x = F.relu(self.fc4(x))
        return x

class Net_2(nn.Module):

    def __init__(self) -> None:
        super(Net, self).__init__()
        self.conv1 = nn.Conv3d(1, 6, 5, 5)
        self.pool = nn.MaxPool3d(2, 2)
        self.conv2 = nn.Conv3d(6, 16, 2, 2)
        self.fc1 = nn.Linear(1152 , 512)
        self.fc2 = nn.Linear(512, 126)
        self.fc3 = nn.Linear(126, 2)
        #self.fc4 = nn.Linear(16, 2)
        self.fcc1=nn.Linear(1024, 256)
        self.fcc2=nn.Linear(256, 4)
        self.fcc1=nn.Linear(4,2)


    def forward(self, x: Tensor, y: Tensor) -> Tensor:
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.shape[0],x.shape[1]*x.shape[2]*x.shape[3]*x.shape[4])
        x1 = F.relu(self.fc1(x))
        x2 = F.relu(self.fc2(x1))
        x3 = F.relu(self.fc3(x2))
       # x = F.relu(self.fc4(x))
        y = self.pool(F.relu(self.conv1(y)))
        y = self.pool(F.relu(self.conv2(y)))
        y = y.view(y.shape[0],x.shape[1]*y.shape[2]*y.shape[3]*y.shape[4])
        y1 = F.relu(self.fc1(y))
        y2 = F.relu(self.fc2(y1))
        y3 = F.relu(self.fc3(y2))

        c1=nn.cat((x1,y1))
        yx1 = F.relu(self.fcc1(c1))
        c2=nn.cat((x2,y2))
        cc2=nn.cat((c2,yx1))
        yx2=F.relu(self.fcc2(cc2))
        c3=nn.cat((x3,y3))
        cc3=nn.cat((c3,yx2))
        yx3=F.relu(self.fcc3(cc3))

        return yx3

class CNNModel(nn.Module):
    def __init__(self,num_classes):
        super(CNNModel, self).__init__()
        chn=1
        self.conv_layer1 = self._conv_layer_set(chn, 32)
        self.conv_layer2 = self._conv_layer_set(32, 64)
        self.conv_layer3 = self._conv_layer_set(64, 128)
        self.fc1 = nn.Linear(200704,1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, num_classes)
        self.relu = nn.LeakyReLU()
        self.batch=nn.BatchNorm1d(1024)
        self.drop=nn.Dropout(p=0.3)

    def _conv_layer_set(self, in_c, out_c):
        conv_layer = nn.Sequential(
        nn.Conv3d(in_c, out_c, kernel_size=(3, 3, 3), padding=0),
        nn.LeakyReLU(),
        nn.MaxPool3d((2, 2, 2)),
        )
        return conv_layer


    def forward(self, x):
        # Set 1
        x=x.float()
        out = self.conv_layer1(x)
        out = self.conv_layer2(out)
        out = self.conv_layer3(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.relu(out)
        if out.shape[0]>1:
            out = self.batch(out)
        out = self.drop(out)
        out = self.fc2(out)
        out = self.fc3(out)
        out = self.fc4(out)
        #print(out.shape)
        return out


class networks:
    def __init__(self,name,device):
        self.net=name
        self.device=device

    def build(self,c=2):
        if self.net=='resnet50':
            net=monai.networks.nets.resnet50(spatial_dims=2,n_input_channels=3,num_classes=c).to(self.device)
        elif self.net=='vit':
            net=monai.networks.nets.ViT(in_channels=3, img_size=(256,256), patch_size=(32,32), proj_type='conv', pos_embed_type='sincos', classification=True,
num_classes=c, spatial_dims=2).to(self.device)
        elif self.net=='densenet121':
            net=monai.networks.nets.DenseNet121(spatial_dims=2, in_channels=3, out_channels=c).to(self.device)
        elif self.net=='seresnext50':
            net=monai.networks.nets.SEResNext50(spatial_dims=3, in_channels=1, out_channels=c).to(self.device)
        elif self.net=='highresnet50':
            net=monai.networks.nets.HighResNet(spatial_dims=3, in_channels=1, out_channels=c).to(self.device)
        elif self.net=='vgg16':
            net = models.vgg16(pretrained=False).to(self.device)
            n_inputs = net.classifier[6].in_features
            net.classifier[6] = nn.Sequential(nn.Linear(n_inputs, 256), nn.ReLU(), nn.Dropout(0.2),nn.Linear(256, c), nn.LogSoftmax(dim=1)).to(self.device)
        else:
            print('nodefine network!!')

        return net

In [ ]:
!pip install torchdata
%pip install vit-pytorch
!pip install einops
!pip install itk

In [ ]:

DATA_ROOT1 = "/content/gdrive/MyDrive/Colab/DATA_cell/A/"
DATA_ROOT2 = "/content/gdrive/MyDrive/Colab/DATA_cell/B/"


import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchdata.datapipes.iter import IterableWrapper
import nibabel as nib
import numpy as np
from scipy import ndimage
import skimage.transform as skTrans
from PIL import Image
from monai.transforms import (
    EnsureChannelFirstd,
    AsDiscrete,
    Compose,
    CropForegroundd,
    EnsureTyped,
    FgBgToIndicesd,
    LoadImaged,
    Orientationd,
    RandCropByPosNegLabeld,
    ScaleIntensityRanged,
    Spacingd,
    RandFlip,
    RandRotate,
    RandZoom,
    RandAffineGrid,
    RandGaussianNoise,
    RandShiftIntensity,
)

def normalize(volume):
    min=np.min(volume)
    max=np.max(volume)
    volume[volume<min]=min
    volume[volume>max]=max
    volume=(volume-min)/(max-min)
    volume=volume.astype("float32")
    return volume

def resize_volume(img,d=84,w=128,h=128):
    """Resize across z-axis"""
    # Set the desired depth
    desired_depth = d
    desired_width = w
    desired_height = h
    # Get current depth
    current_depth = img.shape[2]
    current_width = img.shape[0]
    current_height = img.shape[1]
		# Compute depth factor
    depth = current_depth / desired_depth
    width = current_width / desired_width
    height = current_height / desired_height
    depth_factor = 1 / depth
    width_factor = 1 / width
    height_factor = 1 / height
    #print(img.shape,desired_depth,desired_width,desired_height,width_factor,depth_factor,height_factor)
    if len(img.shape)==4:
        if img.shape[1]==img.shape[2]:
            img=np.transpose(img, (1, 2, 0, 3))
        img = ndimage.zoom(img, (width_factor, height_factor, depth_factor,1), order=1)
    else:
        if img.shape[1]==img.shape[2]:
            img=np.transpose(img, (1, 2, 0))
        img = ndimage.zoom(img, (width_factor, height_factor, depth_factor), order=1)
    return img

class GraphImageDataset(Dataset):
    def __init__(self, data_path:str = "",
                        transform=None,inp=0,sx=128,sy=128,sz=84):
        self.data_path = data_path
        self.transform = transform
        self.inp=inp
        self.sx,self.sy,self.sz=sx,sy,sz

    def __len__(self):
        return len(os.listdir(self.data_path))-1

    def __getitem__(self, index):
        #image_collection.append(image)

        # Load the image
        sub=os.listdir(self.data_path)[index]
        img_from_disk = Image.open(self.data_path+str(sub))
        image = np.array(img_from_disk)
        # Apply the transform to the image
        #print(image.shape)
        image=skTrans.resize(image,(256,256,3),order=1,preserve_range=True)
        if self.transform:
            image = self.transform((image))
            #image = np.squeeze(image)
        image=normalize(image)
        #image=resize_volume(image,d=self.sz,w=self.sx,h=self.sy)
        image=np.transpose(image,(2,0,1))
        #image=skTrans.resize(image,(1,self.sx,self.sy,self.sz),order=1,preserve_range=True)
        #print(image.shape)
        image=torch.tensor(image)
        y=torch.tensor(self.inp)
        return image,y

def load_data(batch_size=1, transform=None) -> Tuple[torch.utils.data.DataLoader, torch.utils.data.DataLoader, Dict]:
    """Load CIFAR-10 (training and test set)."""
    # transform = Compose([RandGaussianNoise(prob=0.1),RandShiftIntensity(prob=0.2,offsets=(10,20)),RandFlip(prob=0.2),RandRotate(prob=0.2,range_x=[0.4,0.4])])
    portion=0.8
    trainset1 = GraphImageDataset(DATA_ROOT1, transform=transform, inp=0)
    trainset2 = GraphImageDataset(DATA_ROOT2, transform=transform, inp=1)
    print(len(trainset1),len(trainset2))
    mid=int((len(trainset1)+len(trainset2))*portion)
    whole=int((len(trainset1)+len(trainset2)))
    print(mid,whole)
    trainset = torch.utils.data.ConcatDataset([trainset1,trainset2])
    gen1 = torch.Generator().manual_seed(whole)
    whole_sampler = torch.utils.data.RandomSampler(trainset,num_samples=whole, generator=gen1)
    indices=list(whole_sampler)
    train_sampler=torch.utils.data.sampler.SubsetRandomSampler(indices[:mid])
    test_sampler=torch.utils.data.sampler.SubsetRandomSampler(indices[mid:])
    print(len(trainset))
    print(list(test_sampler))
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,  sampler=train_sampler)
    testloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,  sampler=test_sampler)
    num_examples = {"trainset" : whole, "testset" : whole-mid}
    return trainloader, testloader, num_examples, trainset, trainset



In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def data_build(bz=1):
    print("Centralized PyTorch training")
    print("Load data")
    trainloader, testloader, _ ,_,_= load_data(batch_size=bz)
    return trainloader, testloader


In [ ]:
!export 'PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb:256'
%pip install "monai--weekly[nibabel, skimage, pillow, tensorboard, gdown, ignite, torchvision, itk, tqdm, lmdb, psutil]"


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import PIL

batch_n = 1

train_loader, validation_loader= data_build(bz=batch_n)

def imshow(img):
    #img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    npimg=np.squeeze(npimg)
    npimg=np.transpose(npimg, (1, 2, 0))
    plt.imshow(npimg)
    plt.imsave(PATH+'image.png',npimg)

dataiter = iter(train_loader)
images, labels = next(dataiter)

# show images
imshow(images)



In [ ]:
import itk

# Load the RGB image
input_image = itk.imread(PATH+'image.png',itk.RGBPixel[itk.UC])
# Define the pixel type and dimension for the image

PixelType = itk.UC  # Unsigned Char for typical image data
RGBPixelType = itk.RGBPixel[PixelType]
Dimension = 2  # Assuming a 2D image

# Define the image type
ImageType = itk.Image[RGBPixelType, Dimension]

# Cast the input image to the defined type
cast_image = itk.CastImageFilter[input_image, ImageType].New(input_image)
cast_image.Update()

# Split the RGB image into its individual channels
r_channel = itk.VectorIndexSelectionCastImageFilter.New(Input=input_image, Index=0)
g_channel = itk.VectorIndexSelectionCastImageFilter.New(Input=input_image, Index=1)
b_channel = itk.VectorIndexSelectionCastImageFilter.New(Input=input_image, Index=2)

r_channel.Update()
g_channel.Update()
b_channel.Update()

# Define the image type
ImageType = itk.Image[PixelType, Dimension]

# Define the Landweber Deconvolution filter type
DeconvolutionFilterType = itk.LandweberDeconvolutionImageFilter[ImageType, ImageType]

# Number of iterations for the deconvolution
number_of_iterations = 50  # Adjust this as needed


# Create a small Gaussian kernel
kernel_size = [5, 5]  # Adjust the size as needed
sigma = 1.0  # Standard deviation of the Gaussian

# Create the image type for the PSF (2D image with float pixel type)
ImageTypeF = itk.Image[itk.F, 2]
ImageTypeUC = itk.Image[itk.UC, 2]

# Create an empty image with the specified size and pixel type
psf = ImageTypeF.New()
psf.SetRegions(itk.Size[2](kernel_size))
psf.Allocate()
psf.FillBuffer(0.0)

# Set a single pixel in the center to 1 (white spot)
index = [int(k // 2) for k in kernel_size]
psf.SetPixel(index, 1.0)

# Apply Gaussian smoothing to create a Gaussian PSF
gaussian_filter = itk.SmoothingRecursiveGaussianImageFilter[ImageTypeF, ImageTypeF].New()
gaussian_filter.SetInput(psf)
gaussian_filter.SetSigma(sigma)
gaussian_filter.Update()

kernel_image_float = gaussian_filter.GetOutput()

# Cast the kernel image to unsigned char type
caster = itk.CastImageFilter[ImageTypeF, ImageTypeUC].New()
caster.SetInput(kernel_image_float)
caster.Update()

kernel_image = caster.GetOutput()

# Set the KernelImage in the deconvolution filter
deconvolve_r = DeconvolutionFilterType.New(Input=r_channel.GetOutput(), KernelImage=kernel_image, NumberOfIterations=number_of_iterations)
deconvolve_g = DeconvolutionFilterType.New(Input=g_channel.GetOutput(), KernelImage=kernel_image, NumberOfIterations=number_of_iterations)
deconvolve_b = DeconvolutionFilterType.New(Input=b_channel.GetOutput(), KernelImage=kernel_image, NumberOfIterations=number_of_iterations)

deconvolve_r.Update()
deconvolve_g.Update()
deconvolve_b.Update()


# Combine the deconvolved channels back into an RGB image
rgb_image_deconvolved = itk.ComposeImageFilter.New(Input1=deconvolve_r.GetOutput(), Input2=deconvolve_g.GetOutput(), Input3=deconvolve_b.GetOutput())
rgb_image_deconvolved.Update()

# Save the result
output_path = PATH+'image_filtered.png'
itk.imwrite(rgb_image_deconvolved.GetOutput(), output_path)


In [ ]:
def tuple_of_tensors_to_tensor(tuple_of_tensors):
    return  torch.stack(list(tuple_of_tensors), dim=0)

import warnings
import os
import wandb
from monai.utils.enums import MetricReduction
from monai.metrics import ConfusionMatrixMetric, ROCAUCMetric
from monai.transforms import (
    AsDiscrete,
    Activations,
)
class AverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = np.where(self.count > 0, self.sum / self.count, self.sum)

warnings.filterwarnings("ignore")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
run_acc = AverageMeter()
run_acc1 = AverageMeter()
run_acc2 = AverageMeter()
run_acc3 = AverageMeter()
run_acc4 = AverageMeter()

roc=ConfusionMatrixMetric(metric_name="accuracy",compute_sample=True,reduction=MetricReduction.MEAN_BATCH, get_not_nans=False)
f1=ConfusionMatrixMetric(metric_name="f1 score",compute_sample=True,reduction=MetricReduction.MEAN_BATCH, get_not_nans=False)
sens=ConfusionMatrixMetric(metric_name="sensitivity",compute_sample=True,reduction=MetricReduction.MEAN_BATCH, get_not_nans=False)
spec=ConfusionMatrixMetric(metric_name="specificity",compute_sample=True,reduction=MetricReduction.MEAN_BATCH, get_not_nans=False)
f1.reset()
spec.reset()
sens.reset()
roc.reset()

batch_n = 1

train_loader, validation_loader= data_build(bz=batch_n)

training=True  #turning it to TRue if you want training. If only testing False

num_epochs = 100
lr = 0.01
name='vit'
#net=CNNModel(num_classes=2).to(DEVICE)
net2=networks(name,device)
#net2=networks('vgg16',device)
#net2=networks('resnet50',device)
#net2=networks('densenet121',device)
net=net2.build(c=2)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=lr)
post_label = AsDiscrete(argmax=True, to_onehot=2)


#wandb.login(key="")


#run = wandb.init(
#    # Set the project where this run will be logged
#    project="ovarian-cancer-project", #"prova-project-ovarian"
#    # Track hyperparameters and run metadata
#    config={
#        "learning_rate": lr,
#        "epochs": num_epochs,
#        "batch" : batch_size
#    })

print_every = 1
min=1000000000

if os.path.exists(PATH+".pth"):
    if torch.cuda.is_available():
        checkpoint = torch.load(PATH+".pth")
        net.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        min = checkpoint['loss']
    else:
        checkpoint = torch.load(PATH+".pth",map_location=torch.device('cpu'))
        net.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        min = checkpoint['loss']
    print('load last saved file: ',PATH, " loss: ",min)

if training==False:
  num_epochs=0

for epoch in range(num_epochs):

    net.train()  # Set the model in training mode
    total_loss_train = 0
    correct_train = 0
    total_train = 0

    for i, data in enumerate(train_loader):
        images=data[0].to(device=device, dtype=torch.float)
        labels=data[1].to(device)

        # Forward pass
        output = net(images)
        #output = torch.stack(list(output), dim=0)
        if name=='vit':
          output2=output[0]
        else:
          output2=output
        output = torch.stack(list(output2), dim=0)
        # Compute loss
        loss = loss_function(output, labels)
        total_loss_train += loss.item()

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate training accuracyfV
        _, predicted = output.max(1)
        correct_train += predicted.eq(labels).sum().item()
        total_train += labels.size(0)


    avg_loss_train = total_loss_train / len(train_loader)
    accuracy_train = correct_train / total_train

    net.eval()  # Set the model in evaluation mode
    total_loss_val = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for i, data in enumerate(validation_loader):
            images=data[0].to(device=device, dtype=torch.float)
            label = data[1].to(device)
            # Forward pass
            output = net(images)
            if name=='vit':
                output2=output[0]
            else:
                output2=output
            output1 = torch.stack(list(output2), dim=0)
            labels=[post_label(out) for out in label]
            output=[post_label(out1) for out1 in output1]
            # Compute loss
            loss = loss_function(output1, label)
            total_loss_val += loss.item()
            roc(y_pred=output, y=labels)
            f1(y_pred=output, y=labels)
            spec(y_pred=output, y=labels)
            sens(y_pred=output, y=labels)
            #print(output,labels)
            # Calculate accuracy
            _, predicted = output1.max(1)
            correct_val += predicted.eq(label).sum().item()
            total_val += label.size(0)
            acc1 = roc.aggregate()
            acc2_total = (f1.aggregate())
            acc4_total = (spec.aggregate())
            acc5_total = (sens.aggregate())
            acc2=torch.nanmean(acc2_total[0])
            acc4=torch.nanmean(acc4_total[0])
            acc5=torch.nanmean(acc5_total[0])
            acc6=torch.nanmean(acc1[0])
            run_acc2.update(acc2.cpu().numpy())
            run_acc1.update(acc6.cpu().numpy())
            run_acc4.update(acc5.cpu().numpy())
            run_acc3.update(acc4.cpu().numpy())
            f1_v = run_acc2.avg
            roc_v= run_acc1.avg
            specif=run_acc3.avg
            scens=run_acc4.avg

    avg_loss_val = total_loss_val / len(validation_loader)
    accuracy_val = correct_val / total_val
    if avg_loss_val<min:
            min=avg_loss_val
            print('save model')
            torch.save({'epoch': epoch,'model_state_dict': net.state_dict(),'optimizer_state_dict': optimizer.state_dict(),'loss':avg_loss_val}, PATH+".pth")

    print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_loss_train:.4f}, Validation Loss: {avg_loss_val:.4f}, Train Accuracy: {accuracy_train:.4f}, Val Accuracy: {accuracy_val:.4f}")
    print("F1: ",f1_v,"ROC: ",roc_v,"Specif: ",specif, "Scen: ",scens)

    #wandb.log({"loss_train": avg_loss_train, "loss_val": avg_loss_val, "acc_train": accuracy_train, "acc_val": accuracy_val})


#wandb.finish()
